This is my first NLP project! I want to write a program that tests Zipf's law. First, I need to create a simple text tokenizer from scratch. Then I will use it to calculate word frequencies. This is the 'word frequency analzyer' part. Then I can use those frequencies to rank the words, and then plot the ranks against the frequencies to prove Zipf's law. Also I can use the Zipf's law formula to predict the frequencies of other words based on their ranks and the frequency of the most frequent word in the text. I can then compare those predicted frequencies with the actual frequencies. 

A Simple Tokenizer

1) Sentence Segmentor:

This program will turn the corpus into a list of sentences, where each sentence is a string that starts with $<s>$ and ends with $</s>$
This particular program can only process .txt files for now :(

In [ ]:
file = input("Enter file name of the .txt file or paste your text directly")
if file.endswith('.txt'):
    fhandle=open(file, 'r', encoding='utf-8')
    text=fhandle.read()
else:
    text=file
print(text)
#testing it 
#update: it worked with the file testing sentence segmenting.txt


Meanwhile, the television drama L.A. Law ran an episode involving a retiree accused of toad licking, and Beavis and Butt-Head licked toads on MTV. (“Tastes like chicken,” Beavis remarked.) Psychedelic toads also captured the attention of lawmakers: A state representative in Georgia cautioned his colleagues about “the extreme dangers of toad licking,” while a representative in South Carolina proposed sentencing offenders to community service at zoos. The men appear to have been speaking in jest, but Canadian police were serious when, in 1991, they urged officials to ban imports of cane toads and outlaw them as pets, citing “the deaths of several Australian drug users.” Before long, Australia also passed laws against psychoactive toad use, listing bufotenine as a controlled substance. Toad licking had become so ingrained in popular consciousness that three decades later, while I was working on this book, people continuously brought it up as a point of reference. Yet the hype about cane t

Re-writing the previous code here instead of modifying the previous cells because I want to document everything!

btw, this is where I got the text from: https://longreads.com/2026/08/04/psychedelic-toads-ego-trip-kimon-de-greef/

In [ ]:
file = input("Enter file name of the .txt file or paste your text directly")
if file.endswith('.txt'):
    fhandle=open(file, 'r', encoding='utf-8')
    text=fhandle.read()
else:
    text=file
sentences = text.split('.')
print(sentences)
#it's working!


['Meanwhile, the television drama L', 'A', ' Law ran an episode involving a retiree accused of toad licking, and Beavis and Butt-Head licked toads on MTV', ' (“Tastes like chicken,” Beavis remarked', ') Psychedelic toads also captured the attention of lawmakers: A state representative in Georgia cautioned his colleagues about “the extreme dangers of toad licking,” while a representative in South Carolina proposed sentencing offenders to community service at zoos', ' The men appear to have been speaking in jest, but Canadian police were serious when, in 1991, they urged officials to ban imports of cane toads and outlaw them as pets, citing “the deaths of several Australian drug users', '” Before long, Australia also passed laws against psychoactive toad use, listing bufotenine as a controlled substance', ' Toad licking had become so ingrained in popular consciousness that three decades later, while I was working on this book, people continuously brought it up as a point of reference', '

Now I need to account for abbreviations like 'Dr.', because they end with a period, but they're not always the end of a sentence. I thought of making a list or set of common abbreviations, but this would need an unncessarily complicated code. I also think it wouldn't be the most efficient with regards to compute. So I think I'm going to use regular expressions! This will give more options than a list or set of common abbreviations. 
Also, I just remembered that sentences can end with ! and ?, so I will account for these as well!
Testing some regex to decide what I will use:

In [2]:
import re
regex = r'\.(?! [a-z])'
test = re.split(regex, 'just testing this regex, here is a period. followed by lowercase letters.')
print(test)

['just testing this regex, here is a period. followed by lowercase letters', '']


I still have that empty string at the end. It seems like an empty string is added every time we split at a period. So let's get rid of empty strings in the list.

In [3]:
test = list(filter(None, test))

In [4]:
if None in test:
    print('still there...')

In [5]:
print(test)

['just testing this regex, here is a period. followed by lowercase letters']


Okay so this worked! 
Now I will need to use a better regex!

So out of the possible cases I can think of, an abbreviation being followed by lowercase letters definitely does NOT indicate the end of a sentence. Example: 'The U.S.A. is a very big country.' Of course it an be followed by a proper name starting with an uppercase letter, as in 'Dr. Jones', but this would require adding more conditions, and maybe keeping a list of proper nouns? 
Anyway, let's try this: always split at a question mark '?' or an exclamation mark '?', but when there is a period '.', only split if there isn't an abbrevation followed by lowercase letters. 
More ideas: I need to account for decimal numbers as well. So I know I shouldn't split the text if there is a period followed immediately by a digit! Also, I figured I should keep the '.', '?', and '!' at the end after splitting. Punctuation is an important part of the corpus. 

Alright I worked this out on paper, and it turned out I will need a very simple regex, which will cover most cases. I should simply split the text if there's a period '.', or a question mark '?', or an exclamation mark '!', all followed by a space, and an uppercase character. This will cover abbreviations that are followed by a lowercase character, and also decimal numbers. Now what's left is abbreviations that are typically followed by an uppercase character, like 'Dr.' and 'Prof.', and for this I think the best way is to keep a set of the most common ones. So now I can build a fucntion that checks these two conditions:
- The text matches the regex

AND

- It doesn't belong to that set of obbreviations!

So the text is split only if both conditions are met!!

Testing the regex.

It seems to be working as intended!

In [ ]:
regex = r'(?<=[.?!]) (?=[A-Z])'
test1 = re.split(regex, 'just testing this regex, here is a period. followed by lowercase letters. Here is a question mark? And here is an exclamation mark!')
print(test)

['just testing this regex, here is a period. followed by lowercase letters.', 'Here is a question mark?', 'And here is an exclamation mark!']


In [23]:
test2 = re.split(regex, 'This is an abbreviation L.A. followed by a lowercase letter. I am 1.6 m tall.')
print(test2)
#it works for decimal numbers too!

['This is an abbreviation L.A. followed by a lowercase letter.', 'I am 1.6 m tall.']


Now let's build the set of abbreviations that are typically followed by an uppercase letter! I 

In [ ]:
regex = re.compile(r'(?<=[.?!]) (?=[A-Z])')
test1 = re.split(regex, 'just testing this regex, here is a period. followed by lowercase letters. Here is a question mark? And here is an exclamation mark!')
print(test)

['just testing this regex, here is a period. followed by lowercase letters.', 'Here is a question mark?', 'And here is an exclamation mark!']


The function isn't working as intended. For some reason the abbreviations are bypassing my if statement, even when they're in the abbreviations list :(

In [43]:
def sentence_segmentor(text):
    abbreviations = ["Mr.", "Mrs.", "Ms.", "Mx.", "Dr.", "Prof.", "Rev.", "Hon.",
    "Fr.", "Sr.", "Br.", "Pres.", "Gov.", "Sen.", "Rep.", "Gen.",
    "Col.", "Maj.", "Capt.", "Cmdr.", "Lt.", "Sgt.", "Cpl.", "Pvt.",
    "Supt.", "Det.", "Ph.D.", "M.D.", "D.D.S.", "D.V.M.", "J.D.", "Ed.D.",
    "B.A.", "B.S.", "B.Sc.", "M.A.", "M.S.", "M.Sc.", "M.B.A.", "Jr.", "Sr.", "St.", "Mt.", "Ft.", "Pt.", "Ave.", "Blvd.", "Rd.", "Ln.", "Hwy.",
    "Pkwy.", "Ct.", "Cir.", "Pl.", "Inc.", "Ltd.", "Corp.", "Co.", "Bros.", "Univ.", "Inst.",
    "Assn.", "Assoc.", "Dept.", "U.S.", "U.S.A.", "U.K.", "D.C.", "Jan.", "Feb.", "Mar.", "Apr.", "Jun.", "Jul.", "Aug.",
    "Sep.", "Sept.", "Oct.", "Nov.", "Dec.", "Vol.", "No.", "Fig.", "Ch.", "Sec.", "Art.", "Eq.", "Tbl.", "Est.", "Ref.", "Reg.", "Misc.", "Approx.", 
    "U.A.E.", "D.R.", "D.R.C."]
    segments = [] # a list of segmented sentences
    start = 0 # index of the start of the next sentence to be segmented
    regex = re.compile(r'([.?!])(\s+)([A-Z])') # the regex compiled as 3 groups
    for match in regex.finditer(text): #iterating through the matches
        end = match.start(2) # the index of the start of group 2, which is the space
        e = match.end(1) # index of the end of group 1 which is (?<=[.?!])
        s = text.rfind(" ", start, end) # index of the last space in the text unil the end of group 1
        leftword = text[s:e] # extracting the word to the left of the space i.e. group 2
        print(leftword)
        if leftword not in abbreviations: # Checking if the word is an abbreviation
            segments.append(text[start:end]) # segmenting the sentence 
            start = match.start(3) # updating the start index of the sentence to be segmented next
    return segments 

text = "Dr. Smith met with Prof. Johnson at St. Mary's Hospital in Washington, D.C., on Jan. 12, 2025. Mr. Brown arrived at 10 a.m. with his colleague, Mrs. Davis, who recently moved from the U.K. to the U.S.A. The meeting included representatives from Inc. and Ltd., as well as officials from the Dept. of Education and the Univ. of California. According to Dr. Williams, the project will begin in N.Y.C. before expanding to L.A., S.F., and St. Louis. Prof. Anderson presented several examples, e.g., statistical models, NLP applications, and machine learning techniques. The report, edited by Dr. Taylor et al., was published in Vol. 3, No. 2, pp. 45-67, in Sept. 2024. Mr. Thompson said the results were promising, but further research is needed, i.e., additional experiments and a larger sample. The team visited Mt. Vernon, Ft. Lauderdale, and St. Petersburg before returning to the main office on Dec. 15. Ms. Wilson, Ph.D., and Dr. Garcia will review the data, compare the results, and submit their findings to the appropriate authorities. The final report is expected by Mar. 1, 2026, etc., although Dr. Smith noted that the deadline may change."
print(sentence_segmentor(text))



 Prof.
 St.
 2025.

 Mrs.
 U.S.A.
 California.
 Dr.
 St.


 techniques.
 Dr.
 2024.

 sample.
 Mt.
 Ft.
 St.
 15.

 Dr.
 authorities.
 Dr.
['Dr.', 'Smith met with Prof.', 'Johnson at St.', "Mary's Hospital in Washington, D.C., on Jan. 12, 2025.", 'Mr.', 'Brown arrived at 10 a.m. with his colleague, Mrs.', 'Davis, who recently moved from the U.K. to the U.S.A.', 'The meeting included representatives from Inc. and Ltd., as well as officials from the Dept. of Education and the Univ. of California.', 'According to Dr.', 'Williams, the project will begin in N.Y.C. before expanding to L.A., S.F., and St.', 'Louis.', 'Prof.', 'Anderson presented several examples, e.g., statistical models, NLP applications, and machine learning techniques.', 'The report, edited by Dr.', 'Taylor et al., was published in Vol. 3, No. 2, pp. 45-67, in Sept. 2024.', 'Mr.', 'Thompson said the results were promising, but further research is needed, i.e., additional experiments and a larger sample.', 'The team visite